# DNA Methylation Analysis with `cytozip`

CLI walkthrough using 9 mouse 3C-seq cells shipped under
`cytozip_example_data/bam/`. All steps are demonstrated as shell
commands (`! czip ...`); benchmark results produced offline by the
scripts under `tests/` are loaded and rendered as tables.

1. Setup + reference `.cz`
2. BAM → `.cz`  (`czip bam_to_cz`)
3. ALLC → `.cz` (`czip allc2cz`) + benchmark table (BAM→ALLC vs BAM→CZ, ALLC→CZ)
4. `view` / `query` (local) + benchmark table (tabix vs czip query)
5. `view` / `query` a remote `.cz`
6. `catcz` + `merge_cz`
7. `cz_to_anndata`

Benchmark
```python
python tests/benchmark_bam_to_cz.py  -j 20
python tests/benchmark_allc_to_cz.py -j 20
python tests/benchmark_query.py
```

In [79]:
import os,sys
import pandas as pd
os.chdir(os.path.expanduser("~/Projects/Github/cytozip/cytozip_example_data"))

## 1. Build the reference `.cz`

The reference holds the genome-wide `(chrom, pos, strand, context)`
axis. Per-cell `.cz` then store only `mc`/`cov` and reuse the
reference's positions, cutting per-cell size by ~5×.


In [80]:
!czip build_ref --help

usage: czip build_ref [-h] -g GENOME [-O OUTPUT] [-p PATTERN] [-j JOBS]
                      [--keep_temp] [--no_delta]

options:
  -h, --help            show this help message and exit
  -g GENOME, --genome GENOME
                        reference genome FASTA (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: hg38_allc.cz)
  -p PATTERN, --pattern PATTERN
                        nucleotide pattern (default: C)
  -j JOBS, --jobs JOBS  number of parallel processes (CPUs) (default: 12)
  --keep_temp           keep temp directory (default: False)
  --no_delta            disable DELTA encoding on the pos column (default: on,
                        gives ~3x smaller reference files with mild query
                        overhead) (default: False)


In [3]:
!time czip build_ref -g ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O output/mm10_with_chrL.allc.cz -j 20

2026-04-26 13:18:07.418 | DEBUG    | cytozip.allc:WriteC:60 - chr1
2026-04-26 13:18:08.102 | DEBUG    | cytozip.allc:WriteC:60 - chr10
2026-04-26 13:18:08.772 | DEBUG    | cytozip.allc:WriteC:60 - chr11
2026-04-26 13:18:09.438 | DEBUG    | cytozip.allc:WriteC:60 - chr12
2026-04-26 13:18:10.071 | DEBUG    | cytozip.allc:WriteC:60 - chr13
2026-04-26 13:18:10.770 | DEBUG    | cytozip.allc:WriteC:60 - chr14
2026-04-26 13:18:11.307 | DEBUG    | cytozip.allc:WriteC:60 - chr15
2026-04-26 13:18:11.827 | DEBUG    | cytozip.allc:WriteC:60 - chr16
2026-04-26 13:18:12.356 | DEBUG    | cytozip.allc:WriteC:60 - chr17
2026-04-26 13:18:12.875 | DEBUG    | cytozip.allc:WriteC:60 - chr18
2026-04-26 13:18:13.151 | DEBUG    | cytozip.allc:WriteC:60 - chr1_GL456210_random
2026-04-26 13:18:13.153 | DEBUG    | cytozip.allc:WriteC:60 - chr1_GL456211_random
2026-04-26 13:18:13.155 | DEBUG    | cytozip.allc:WriteC:60 - chr19
2026-04-26 13:18:13.156 | DEBUG    | cytozip.allc:WriteC:60 - chr1_GL456212_random
2026

In [26]:
!czip header -I output/mm10_with_chrL.allc.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  1324830611
message  :  /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
formats  :  ['Q', 'c', '3s']
columns  :  ['pos', 'strand', 'context']
sort_col  :  0
delta_cols  :  [0]
chunk_dims  :  ['chrom']
header_size  :  102


In [27]:
! czip view -I output/mm10_with_chrL.allc.cz --show_dims 0 | head

chrom	pos	strand	context
chr1	3000003	+	CTG
chr1	3000005	-	CAG
chr1	3000009	+	CTA
chr1	3000016	-	CAA
chr1	3000018	-	CAC
chr1	3000019	-	CCA
chr1	3000023	+	CTT
chr1	3000027	-	CAA
chr1	3000029	-	CTC


In [28]:
! czip summary -I output/mm10_with_chrL.allc.cz | head

chrom	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	102	95328851	95386814	3615	78962721
chr10	95386814	63310564	158735944	2409	52609184
chr11	158735944	60770669	219544747	2382	52027265
chr12	219544747	58457323	278037836	2234	48799752
chr13	278037836	58621213	336694783	2232	48750883
chr14	336694783	60350467	397081896	2289	49987736
chr15	397081896	50425550	447538412	1934	42230765
chr16	447538412	47066788	494633718	1781	38899643
chr17	494633718	46270276	540932704	1793	39153472


## 3. BAM → `.cz`

Convert a position-sorted BAM directly to `.cz` (no intermediate ALLC
text). The output has only `mc` / `cov` because we pass a reference.


In [81]:
!czip bam_to_cz --help

usage: czip bam_to_cz [-h] -I INPUT -g GENOME [-O OUTPUT]
                      [--num_upstr_bases NUM_UPSTR_BASES]
                      [--num_downstr_bases NUM_DOWNSTR_BASES]
                      [--min_mapq MIN_MAPQ]
                      [--min_base_quality MIN_BASE_QUALITY] [-c BATCH_SIZE]
                      [--convert_bam_strandness] [--save_count_df]
                      [--mode {full,pos_mc_cov,mc_cov}]
                      [--count_fmt {B,H,I,Q}] [-r REFERENCE]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input position-sorted BAM (bismark/hisat-3n) (default:
                        None)
  -g GENOME, --genome GENOME
                        indexed reference fasta (.fai required) (default:
                        None)
  -O OUTPUT, --output OUTPUT
                        output .cz path (default: <bam_stem>.cz) (default:
                        None)
  --num_upstr_bases NUM_UPSTR_BASES
              

In [82]:
! time czip bam_to_cz -I bam/FC_M_P12b_3C_2-5-M17-N10.hisat3n_dna.all_reads.deduped.bam \
     --genome ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O output/cz/FC_M_P12b_3C_2-5-M17-N10.cz \
     --mode mc_cov --count_fmt B --reference output/mm10_with_chrL.allc.cz

/home/x-wding2/Software/conda/m3c/lib/python3.10/site-packages/cytozip/bam.py:719: UserWarning: mc/cov value exceeds count_fmt='B' max (255); clipping. Consider count_fmt='H' for bulk/high-coverage data.
  _handle_site(fields[0], pos0, ref_base, unconverted, converted)

real	3m37.585s
user	4m0.875s
sys	0m10.991s


In [31]:
#bam to allc
from ALLCools._bam_to_allc import bam_to_allc
%time bam_to_allc(bam_path="bam/FC_M_P12b_3C_2-5-M17-N10.hisat3n_dna.all_reads.deduped.bam", \
                  reference_fasta=os.path.expanduser("~/Ref/mm10/mm10_ucsc_with_chrL.fa"), \
                  output_path="output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz", \
                  cpu=1,num_upstr_bases=0, num_downstr_bases=2, \
                  min_mapq=10, min_base_quality=20)

CPU times: user 2min 58s, sys: 7.24 s, total: 3min 5s
Wall time: 3min 21s


,mc,cov,mc_rate,genome_cov
CTT,2135943,4141584,0.515731,0.066954
CTA,1326142,2710024,0.489347,0.066954
CAA,1716819,3719684,0.461550,0.066954
CCA,1680893,3521141,0.477372,0.066954
CAC,1501706,3073867,0.488540,0.066954
CAG,1929009,4047201,0.476628,0.066954
CTG,2026603,4048731,0.500553,0.066954
CCT,1706829,3381003,0.504829,0.066954
CCC,1108042,2312639,0.479124,0.066954
CAT,1831117,3671417,0.498749,0.066954


### BAM → `.cz` benchmark vs ALLCools

Produced by `tests/benchmark_bam_to_cz.py -j 9`. ALLCools writes a
gzipped TSV (`.allc.tsv.gz`); cytozip writes a chunked, columnar
zstd-compressed `.cz`.


In [32]:
import pandas as pd
bam_bench = pd.read_csv('output/bam_benchmark/bam_benchmark.tsv', sep='\t')
bam_bench.round(2)

,cell,bam_size_mb,n_reads,allc_wall_s,allc_rss_mb,allc_size_mb,allc_cached,cz_wall_s,cz_rss_mb,cz_size_mb,speedup,size_ratio
0,FC_E17b_3C_5-5-I24-A21,685.99,7330382,1566.53,571.61,442.62,True,474.25,970.04,78.90,3.30,0.18
1,FC_M_E15a_3C_1-1-I5-B1,177.37,1787812,728.97,571.79,102.53,True,130.61,984.11,22.41,5.58,0.22
2,FC_M_P12b_3C_2-5-M17-N10,239.89,2307326,884.07,577.01,143.12,True,175.36,981.99,29.85,5.04,0.21
3,FC_M_P3b_3C_6-6-J3-P24,147.36,1395949,628.37,573.83,85.71,True,115.81,983.71,19.52,5.43,0.23
4,FC_M_P6a_3C_7-3-K21-P5,89.76,938870,412.18,574.35,48.84,True,78.38,985.21,12.32,5.26,0.25
5,FC_M_P9B_3C_6-2-F6-O4,32.28,342778,245.17,572.93,17.61,True,47.51,967.83,6.15,5.16,0.35
6,FC_P0b_3C_5-1-I24-J14,485.51,5028030,1385.09,574.75,285.50,True,308.73,974.92,52.72,4.49,0.18
7,FC_P13a_3C_7-1-A11-O1,260.71,2588754,949.58,571.56,155.84,True,183.12,980.34,31.47,5.19,0.20
8,FC_P28a_3C_2-1-E5-N14,186.32,1908846,803.77,574.15,112.95,True,139.57,983.42,24.11,5.76,0.21


In [33]:
!cat output/bam_benchmark/bam_benchmark.txt

cytozip bam_to_cz  vs  ALLCools bam_to_allc
reference FASTA : /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
reference .cz   : /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/mm10_with_chrL.allc.cz
BAMs            : 9   total reads = 23,628,747
ALLCools cached : 9/9 (timing reused from disk)

ALLCools : time=  7603.7 s   peak_rss=  577.0 MB   size=  1394.7 MB
cytozip  : time=  1653.3 s   peak_rss=  985.2 MB   size=   277.5 MB

speedup (allc / cz time)  =  4.60x
compression (cz / allc)   =  19.9%


## 4. ALLC → `.cz`

Pack an existing `.allc.tsv.gz` into `.cz`. Two layouts:

In [34]:
! czip allc2cz --help

usage: czip allc2cz [-h] -I INPUT -O OUTPUT [-r REFERENCE]
                    [--missing_value MISSING_VALUE] [-F FORMATS] [-C COLUMNS]
                    [-D CHUNK_DIMS] [-u USECOLS] [--ref_pos_col REF_POS_COL]
                    [--allc_pos_col ALLC_POS_COL] [-s SEP]
                    [--chrom_order CHROM_ORDER] [-c BATCH_SIZE]
                    [--sort_col SORT_COL] [--delta_cols DELTA_COLS] [-j JOBS]
                    [--pattern PATTERN] [--no_skip_existing]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input allc.tsv.gz, OR a directory containing many
                        allc.tsv.gz (batch mode: --output must be a directory)
                        (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (single-file), or output directory
                        (batch mode) (default: None)
  -r REFERENCE, --reference REFERENCE
                        reference .cz file (

In [83]:
# Standalone (positions inside the .cz)
! time czip allc2cz -I output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz \
      -O output/allc/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz \
      -F Q,B,B -C pos,mc,cov -u 1,4,5

2026-04-26 19:52:58.868 | INFO     | cytozip.allc:allc2cz:256 - output/allc/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz existed, skip.

real	0m0.398s
user	0m0.072s
sys	0m0.041s


In [84]:
# Reference-aligned (positions come from the reference .cz)
! time czip allc2cz -I output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz \
      -O output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
      -r output/mm10_with_chrL.allc.cz

2026-04-26 19:52:59.123 | INFO     | cytozip.allc:allc2cz:256 - output/allc/FC_M_P12b_3C_2-5-M17-N10.cz existed, skip.

real	0m0.108s
user	0m0.074s
sys	0m0.030s


In [85]:
! ls output/allc/FC_M_P12b_3C_2-5-M17-N10* -sh

 512 output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.bench.json
137M output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz
1.5M output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz.tbi
 29M output/allc/FC_M_P12b_3C_2-5-M17-N10.cz
 42M output/allc/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz


## 5. `view` and `query`

`czip view` streams an entire dimension (e.g. one chrom). `czip query`
pulls a region. Both can use `-r REFERENCE` to expand reference-aligned
files into full ALLC-style records (chrom / pos / context / mc / cov).


In [86]:
! czip view -I output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz --show_dims 0 | awk '$6>0' | head

chrom	pos	strand	context	mc	cov
chr1	3005823	+	CTT	0	1
chr1	3005826	+	CTA	0	1
chr1	3005836	+	CAA	0	1
chr1	3005840	+	CCA	0	1
chr1	3005841	+	CAC	0	1
chr1	3005843	+	CCA	0	1
chr1	3005844	+	CAC	0	1
chr1	3005846	+	CAA	1	1
chr1	3005850	+	CCA	0	1


In [87]:
! tabix output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz chr9:3000294-3005294 | head

chr9	3000294	-	CAT	15	23	1
chr9	3000296	+	CCT	51	62	1
chr9	3000297	+	CTA	53	63	1
chr9	3000300	+	CAA	52	65	1
chr9	3000304	-	CAT	2	26	1
chr9	3000305	-	CCA	1	27	1
chr9	3000307	+	CAT	60	71	1
chr9	3000312	+	CTA	63	76	1
chr9	3000321	+	CCA	67	78	1
chr9	3000322	+	CAA	65	76	1
Failed to write to stdout: Broken pipe


In [88]:
! time czip query -I output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz -K chr9 -s 3000294 -e 3005294 | head

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	15	23
chr9	3000296	+	CCT	51	62
chr9	3000297	+	CTA	53	63
chr9	3000300	+	CAA	52	65
chr9	3000304	-	CAT	2	26
chr9	3000305	-	CCA	1	27
chr9	3000307	+	CAT	60	71
chr9	3000312	+	CTA	63	76
chr9	3000321	+	CCA	67	78

real	0m0.250s
user	0m0.136s
sys	0m0.097s


## 6. View / query a remote `.cz` (no download)

cytozip reads `.cz` files directly over HTTP Range requests when they
carry a chunk index. Below we query a remote `.cz` hosted on figshare —
only the needed chunks are fetched on-demand.


In [47]:
! czip header -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  29849112
message  :  mm10_with_chrL.allc.cz
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_dims  :  ['chrom']
header_size  :  61


In [48]:
# view a cz file (no coordinates were stored) alone
! czip view -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz --show_dims 0 | head

chrom	mc	cov
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0


In [49]:
# query remote .cz file with local reference .cz:
! time czip query -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz -K chr9 \
    -s 3000294 -e 3005294 | head -n 5

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	15	23
chr9	3000296	+	CCT	51	62
chr9	3000297	+	CTA	53	63
chr9	3000300	+	CAA	52	65

real	0m1.069s
user	0m0.202s
sys	0m0.129s


In [50]:
# or both the cz and the reference can be accessed via HTTP(S) URLs, without downloading any files locally:
! time czip query -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r https://neomorph.salk.edu/ftp/bican/mm10_with_chrL.allc.cz \
    -K chr9 -s 3000294 -e 3005294 | head -n 5
# query 5000 bp of regions containing 1759 records

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	15	23
chr9	3000296	+	CCT	51	62
chr9	3000297	+	CTA	53	63
chr9	3000300	+	CAA	52	65

real	0m2.147s
user	0m0.236s
sys	0m0.139s


## 7. `catcz` and `merge_cz`

* `catcz` — concatenate per-cell `.cz` into one multi-cell `.cz` by
  adding a `cell_id` dimension.
* `merge_cz` — sum `mc` / `cov` across cells (pooled pseudobulk).


In [51]:
! ls output/cz/*.cz -sh

 76M output/cz/FC_E17b_3C_5-5-I24-A21.cz
 22M output/cz/FC_M_E15a_3C_1-1-I5-B1.cz
 29M output/cz/FC_M_P12b_3C_2-5-M17-N10.cz
 19M output/cz/FC_M_P3b_3C_6-6-J3-P24.cz
 12M output/cz/FC_M_P6a_3C_7-3-K21-P5.cz
6.0M output/cz/FC_M_P9B_3C_6-2-F6-O4.cz
 51M output/cz/FC_P0b_3C_5-1-I24-J14.cz
 31M output/cz/FC_P13a_3C_7-1-A11-O1.cz
 23M output/cz/FC_P28a_3C_2-1-E5-N14.cz


In [56]:
! time czip catcz -O output/all_cells.cz -I "output/cz/*.cz" --key_added cell_id -F B,B -C mc,cov


real	0m0.596s
user	0m0.214s
sys	0m0.349s


In [57]:
! czip header -I output/all_cells.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  277483841
message  :  
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_dims  :  ['chrom', 'cell_id']
header_size  :  47


In [58]:
! czip summary -I output/all_cells.cz | head

chrom	cell_id	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	FC_E17b_3C_5-5-I24-A21	47	6002323	6007238	603	78962721
chr10	FC_E17b_3C_5-5-I24-A21	6007238	4000072	10010571	402	52609184
chr11	FC_E17b_3C_5-5-I24-A21	10010571	3835811	13849603	397	52027265
chr12	FC_E17b_3C_5-5-I24-A21	13849603	3618669	17471301	373	48799752
chr13	FC_E17b_3C_5-5-I24-A21	17471301	3703968	21178290	372	48750883
chr14	FC_E17b_3C_5-5-I24-A21	21178290	3678045	24859436	382	49987736
chr15	FC_E17b_3C_5-5-I24-A21	24859436	3173135	28035200	323	42230765
chr16	FC_E17b_3C_5-5-I24-A21	28035200	2939329	30976950	297	38899643
chr17	FC_E17b_3C_5-5-I24-A21	30976950	2877521	33856908	299	39153472


In [59]:
# merge 9 single-cell .cz files into a pseudobulk .cz file, summing the mc and cov values across all cells:
! time czip merge_cz -i output/cz -O output/pseudobulk.cz \
    -r output/mm10_with_chrL.allc.cz -F H,H -j 20

2026-04-26 14:53:52.754 | INFO     | cytozip.merge:merge_cz:492 - output/pseudobulk.cz
2026-04-26 14:54:01.257 | INFO     | cytozip.merge:_bg_rmtree:141 - Removing temp dir /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/pseudobulk.cz.tmp (in background)

real	0m8.717s
user	1m39.776s
sys	0m4.928s


In [60]:
! czip view -I output/pseudobulk.cz --show_dims 0 \
    -r output/mm10_with_chrL.allc.cz | awk '$6 >10' | head

chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	11	11
chr1	25520458	-	CCA	11	11
chr1	25520463	-	CAG	11	11
chr1	25520464	-	CCA	11	11
chr1	25520474	-	CAG	13	13
chr1	25520479	-	CGG	13	13
chr1	25520480	-	CCG	13	13
chr1	25520481	-	CCC	12	13
chr1	25520482	-	CCC	12	12


In [62]:
! czip query -I output/pseudobulk.cz \
    -r output/mm10_with_chrL.allc.cz -K chr1 -s 25520457 -e 25520482

chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	11	11
chr1	25520458	-	CCA	11	11
chr1	25520459	+	CCC	2	10
chr1	25520460	+	CCT	2	10
chr1	25520461	+	CTG	0	10
chr1	25520463	-	CAG	11	11
chr1	25520464	-	CCA	11	11
chr1	25520465	+	CGT	7	9
chr1	25520466	-	CGC	7	7
chr1	25520469	+	CCC	1	10
chr1	25520470	+	CCC	0	10
chr1	25520471	+	CCT	0	10
chr1	25520472	+	CTG	1	10
chr1	25520474	-	CAG	13	13
chr1	25520477	+	CCG	0	9
chr1	25520478	+	CGG	5	10
chr1	25520479	-	CGG	13	13
chr1	25520480	-	CCG	13	13
chr1	25520481	-	CCC	12	13
chr1	25520482	-	CCC	12	12


In [63]:
! for file in `ls output/cz`; do echo ${file} && czip query -I output/cz/${file} -r output/mm10_with_chrL.allc.cz -K chr1 -s 25520457 -e 25520482; done;

FC_E17b_3C_5-5-I24-A21.cz
chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	3	3
chr1	25520458	-	CCA	3	3
chr1	25520459	+	CCC	2	2
chr1	25520460	+	CCT	2	2
chr1	25520461	+	CTG	0	2
chr1	25520463	-	CAG	3	3
chr1	25520464	-	CCA	3	3
chr1	25520465	+	CGT	1	1
chr1	25520466	-	CGC	1	1
chr1	25520469	+	CCC	1	2
chr1	25520470	+	CCC	0	2
chr1	25520471	+	CCT	0	2
chr1	25520472	+	CTG	1	2
chr1	25520474	-	CAG	3	3
chr1	25520477	+	CCG	0	2
chr1	25520478	+	CGG	1	2
chr1	25520479	-	CGG	3	3
chr1	25520480	-	CCG	3	3
chr1	25520481	-	CCC	3	3
chr1	25520482	-	CCC	3	3
FC_M_E15a_3C_1-1-I5-B1.cz
chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	1	1
chr1	25520458	-	CCA	1	1
chr1	25520459	+	CCC	0	1
chr1	25520460	+	CCT	0	1
chr1	25520461	+	CTG	0	1
chr1	25520463	-	CAG	1	1
chr1	25520464	-	CCA	1	1
chr1	25520465	+	CGT	0	1
chr1	25520466	-	CGC	1	1
chr1	25520469	+	CCC	0	1
chr1	25520470	+	CCC	0	1
chr1	25520471	+	CCT	0	1
chr1	25520472	+	CTG	0	1
chr1	25520474	-	CAG	2	2
chr1	25520477	+	CCG	0	0
chr1	25520478	+	CGG	0	1
chr1	25520479	-	CGG	

## 8. Build a cell × gene `AnnData`

`cytozip.features.cz_to_anndata` aggregates per-cell `.cz` files over
a feature interval set. When `features=` is a GTF path, cytozip
extracts one interval per gene, merges GENCODE records sharing a
symbol, and extends each interval by `flank_bp` (default 2 kb) on
both sides.


In [64]:
! time czip cz_to_anndata -I output/cz \
    -f /home/x-wding2/Ref/mm10/annotations/gencode.vM23.annotation.gtf \
    -O output/allcells.h5ad -r output/mm10_with_chrL.allc.cz -j 10


real	1m30.595s
user	2m50.068s
sys	1m45.009s


In [65]:
import anndata
adata=anndata.read_h5ad("output/allcells.h5ad")
adata

AnnData object with n_obs × n_vars = 9 × 55228
    obs: 'alpha', 'beta', 'prior_mean'
    var: 'chrom', 'start', 'end', 'gene_id', 'gene_name', 'gene_type', 'strand'
    uns: 'cytozip_score'
    layers: 'cov', 'mc'

In [66]:
adata.obs

,alpha,beta,prior_mean
FC_E17b_3C_5-5-I24-A21,16.553249,4.299189,0.793828
FC_M_E15a_3C_1-1-I5-B1,2.186407,2.139431,0.505430
FC_M_P12b_3C_2-5-M17-N10,2.995748,2.875902,0.510205
FC_M_P3b_3C_6-6-J3-P24,1.944791,1.913017,0.504118
FC_M_P6a_3C_7-3-K21-P5,1.220254,1.223011,0.499436
FC_M_P9B_3C_6-2-F6-O4,0.855689,0.833260,0.506640
FC_P0b_3C_5-1-I24-J14,5.304123,5.270100,0.501609
FC_P13a_3C_7-1-A11-O1,2.719193,2.712647,0.500602
FC_P28a_3C_2-1-E5-N14,2.280774,2.224179,0.506281


In [67]:
adata.var

,chrom,start,end,gene_id,gene_name,gene_type,strand
name,,,,,,,
0610005C13Rik,chr7,45565793,45577327,ENSMUSG00000109644.1,0610005C13Rik,lncRNA,-
0610006L08Rik,chr7,74816817,74855813,ENSMUSG00000108652.1,0610006L08Rik,lncRNA,-
0610009B22Rik,chr11,51683385,51690874,ENSMUSG00000007777.9,0610009B22Rik,protein_coding,-
0610009E02Rik,chr2,26443695,26461390,ENSMUSG00000086714.1,0610009E02Rik,lncRNA,+
0610009L18Rik,chr11,120346677,120353190,ENSMUSG00000043644.4,0610009L18Rik,lncRNA,+
...,...,...,...,...,...,...,...
n-R5s96,chr8,47717245,47721369,ENSMUSG00000064508.1,n-R5s96,rRNA,+
n-R5s97,chr8,47897181,47901294,ENSMUSG00000064519.1,n-R5s97,rRNA,+
n-R5s98,chr8,54864260,54868379,ENSMUSG00000084498.1,n-R5s98,rRNA,+


In [68]:
adata.to_df()

name,0610005C13Rik,0610006L08Rik,0610009B22Rik,0610009E02Rik,0610009L18Rik,0610010F05Rik,0610010K14Rik,0610012D04Rik,0610012G03Rik,0610025J13Rik,...,n-R5s90,n-R5s92,n-R5s93,n-R5s94,n-R5s95,n-R5s96,n-R5s97,n-R5s98,n-TSaga9,n-TStga1
FC_E17b_3C_5-5-I24-A21,0.825397,0.802066,0.800847,0.786429,0.873874,0.800176,0.764977,0.812500,0.724036,0.768344,...,0.868852,0.830882,0.813953,0.772152,0.674359,0.748447,0.850467,0.894231,0.888889,0.857143
FC_M_E15a_3C_1-1-I5-B1,0.474820,0.510067,0.558333,0.445902,1.000000,0.477361,0.421053,0.430556,0.926829,0.470437,...,0.705882,0.600000,0.000000,0.000000,1.000000,0.000000,0.000000,0.548387,0.309942,0.000000
FC_M_P12b_3C_2-5-M17-N10,0.618785,0.466392,0.565574,0.552511,0.412698,0.495077,0.426396,0.615385,0.014925,0.555556,...,0.000000,0.369231,1.000000,0.107143,0.377358,0.319444,0.000000,0.491228,0.453488,0.000000
FC_M_P3b_3C_6-6-J3-P24,0.722222,0.510549,0.000000,0.491803,0.714286,0.587121,0.000000,0.000000,0.444444,0.403727,...,0.000000,0.000000,0.000000,0.526316,0.661538,0.000000,0.822222,0.000000,0.313433,0.393701
FC_M_P6a_3C_7-3-K21-P5,1.000000,0.610465,0.565217,0.380682,0.066667,0.599265,1.000000,0.000000,0.037037,1.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.850000,0.385714,0.000000,0.000000,0.000000
FC_M_P9B_3C_6-2-F6-O4,0.000000,0.324176,0.000000,0.473118,0.142857,0.596273,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.157895,0.000000,0.000000,1.000000
FC_P0b_3C_5-1-I24-J14,0.406542,0.514163,0.522422,0.612132,0.848740,0.562095,0.652778,0.372263,0.577947,0.560440,...,0.655172,0.517857,0.551020,0.428571,0.290323,0.546392,0.258883,0.453659,0.303279,0.876712
FC_P13a_3C_7-1-A11-O1,0.725564,0.474082,0.757143,0.404545,0.335484,0.522672,0.386293,0.423729,0.252033,0.515306,...,0.000000,0.276786,0.615894,0.174603,0.000000,0.526786,0.258621,0.800000,0.800000,0.425532
FC_P28a_3C_2-1-E5-N14,0.315789,0.401980,0.612500,0.341772,0.304348,0.385257,1.000000,0.000000,0.547170,0.628205,...,0.270270,0.766234,1.000000,0.707317,0.000000,0.000000,1.000000,0.000000,0.000000,0.434343
